# Lecture 6 — Exact Diagonalisation II: Diagnostics

---

## Overview

Lecture 05 explained how to build the Hamiltonian and find its eigenstates. This lecture focuses on what to *do* with those eigenstates — the observables and diagnostics that reveal the physics of the quantum phase transition.

We cover the energy spectrum, ground-state expectation values, correlation functions, the transverse susceptibility, and how to connect ED data to the finite-size scaling framework from lecture 02. By the end, you will be able to map out the full phase diagram of the TFIM from ED data alone.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})


def build_tfim(N, J=1.0, Gamma=1.0):
    """TFIM Hamiltonian as a sparse CSR matrix (open boundary conditions)."""
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        diag = 0.0
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j
        if diag != 0.0:
            rows.append(state); cols.append(state); data.append(diag)
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(-Gamma)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))


def sz_site(i, N):
    """Sparse diagonal operator for sigma_z at site i."""
    diag = np.array([1 if (s >> i) & 1 else -1 for s in range(2**N)], dtype=float)
    return sp.diags(diag, format='csr')

## 1. The Energy Spectrum

The full set of eigenvalues reveals the structure of the phase diagram through **level spectroscopy**: the gap between the ground state and first excited state closes at the critical point.

- **Ordered phase** ($\Gamma < \Gamma_c$): two nearly degenerate lowest levels, gap $\sim e^{-N/\xi}$.
- **Critical point**: gap closes as $\Delta \sim L^{-z}$.
- **Disordered phase** ($\Gamma > \Gamma_c$): unique ground state, finite gap.

---

In [ ]:
# Level spectroscopy: plot lowest 4 energy levels vs Gamma/J
N = 10
gammas = np.linspace(0.2, 2.0, 60)
n_levels = 4
levels = np.zeros((len(gammas), n_levels))

for k, g in enumerate(gammas):
    H = build_tfim(N, J=1.0, Gamma=g)
    evals = eigsh(H, k=n_levels, which='SA', return_eigenvectors=False)
    levels[k] = np.sort(evals)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
for n in range(n_levels):
    ax.plot(gammas, levels[:, n], color=colors[n], label=f'$E_{n}$')
ax.axvline(1.0, color='black', linestyle='--', alpha=0.5, label=r'$\Gamma_c=J$')
ax.set_xlabel(r'$\Gamma/J$')
ax.set_ylabel('Energy')
ax.set_title(f'Level spectroscopy — N={N}')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Ground-State Expectation Values

Given $|E_0\rangle$, we compute:

- **Order parameter:** $m = N^{-1}\sum_i \langle E_0 | \hat{\sigma}^z_i | E_0 \rangle$ — zero on finite systems with $\mathbb{Z}_2$ symmetry; use $\langle m^2 \rangle$ instead.
- **Transverse magnetization:** $m_x = N^{-1}\sum_i \langle E_0 | \hat{\sigma}^x_i | E_0 \rangle$ — non-zero in both phases.
- **Energy per site:** $e_0 = E_0/N$.

---

In [ ]:
N = 12
gammas_dense = np.linspace(0.1, 3.0, 80)
e0_list, m2_list, mx_list = [], [], []

# Build sigma_x total (for transverse magnetization)
def sx_total(N):
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(1.0)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim)) / N

Mz_total = sum(sz_site(i, N) for i in range(N)) / N
Mx_total = sx_total(N)

for g in gammas_dense:
    H = build_tfim(N, J=1.0, Gamma=g)
    evals, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]
    e0_list.append(evals[0] / N)
    m2_list.append(float((psi0 @ Mz_total @ Mz_total @ psi0).real))
    mx_list.append(float((psi0 @ Mx_total @ psi0).real))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(gammas_dense, e0_list)
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.6)
axes[0].set_xlabel(r'$\Gamma/J$'); axes[0].set_ylabel(r'$E_0/N$')
axes[0].set_title('Energy per site')

axes[1].plot(gammas_dense, m2_list)
axes[1].axvline(1.0, color='red', linestyle='--', alpha=0.6)
axes[1].set_xlabel(r'$\Gamma/J$'); axes[1].set_ylabel(r'$\langle m^2 \rangle$')
axes[1].set_title(r'Order parameter $\langle m^2 \rangle$')

axes[2].plot(gammas_dense, mx_list)
axes[2].axvline(1.0, color='red', linestyle='--', alpha=0.6)
axes[2].set_xlabel(r'$\Gamma/J$'); axes[2].set_ylabel(r'$\langle m^x \rangle$')
axes[2].set_title('Transverse magnetization')

for ax in axes:
    ax.set_title(ax.get_title(), pad=8)
plt.tight_layout()
plt.show()

## 3. Correlation Functions

The spin-spin correlation function $C(r) = \langle E_0 | \hat{\sigma}^z_0 \hat{\sigma}^z_r | E_0 \rangle$ reveals the spatial structure:

- **Ordered phase:** $C(r) \to \text{const} > 0$ — long-range order.
- **Disordered phase:** $C(r) \sim e^{-r/\xi}$ — exponential decay.
- **Critical point:** $C(r) \sim r^{-\eta}$ with $\eta = 1/4$ for the 1D TFIM.

---

In [ ]:
N = 16
phases = [
    (0.3, 'Ordered ($\\Gamma/J=0.3$)', 'tab:blue'),
    (1.0, 'Critical ($\\Gamma/J=1.0$)', 'tab:red'),
    (2.5, 'Disordered ($\\Gamma/J=2.5$)', 'tab:green'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for g, label, color in phases:
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]

    Sz0 = sz_site(0, N)
    Sz0_psi = Sz0 @ psi0

    rs = np.arange(1, N // 2 + 1)
    C = []
    for r in rs:
        Szr = sz_site(r, N)
        c = float((psi0 @ Sz0 @ Szr @ psi0).real)
        C.append(c)

    axes[0].plot(rs, C, 'o-', color=color, label=label, markersize=5)
    # Log scale for decay
    C_abs = np.abs(C)
    axes[1].semilogy(rs, C_abs + 1e-12, 'o-', color=color, label=label, markersize=5)

for ax in axes:
    ax.set_xlabel('$r$')
    ax.legend(fontsize=9)

axes[0].set_ylabel(r'$C(r) = \langle\sigma^z_0 \sigma^z_r\rangle$')
axes[0].set_title('Spin-spin correlations')
axes[1].set_ylabel(r'$|C(r)|$ (log scale)')
axes[1].set_title('Log-scale: ordered vs exponential vs power-law')

# Add power-law reference for critical case
rs_ref = np.linspace(1, N//2, 100)
axes[1].plot(rs_ref, 0.3 * rs_ref**(-0.25), 'k--', alpha=0.5, label=r'$r^{-1/4}$')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. The Transverse Susceptibility

The response of $z$-magnetization to a perturbation, via second-order perturbation theory:

$$\chi = \frac{1}{N} \sum_{n \neq 0} \frac{|\langle E_n | \hat{M}^z | E_0 \rangle|^2}{E_n - E_0}$$

At the QCP, $\chi$ diverges with exponent $\gamma = 7/4$. On finite systems, the peak scales as $\chi_{\max}(L) \sim L^{\gamma/\nu} = L^{7/4}$.

---

In [ ]:
def transverse_susceptibility(N, J, Gamma, n_states=50):
    """Compute susceptibility via sum over excited states (truncated)."""
    H = build_tfim(N, J=J, Gamma=Gamma)
    k = min(n_states, 2**N - 1)
    evals, evecs = eigsh(H, k=k, which='SA')
    idx = np.argsort(evals)
    evals, evecs = evals[idx], evecs[:, idx]

    Mz = sum(sz_site(i, N) for i in range(N))
    psi0 = evecs[:, 0]
    E0 = evals[0]

    chi = 0.0
    for n in range(1, k):
        matrix_element = float((evecs[:, n] @ Mz @ psi0).real)
        chi += matrix_element**2 / (evals[n] - E0)
    return chi / N


sizes = [6, 8, 10, 12]
gammas_chi = np.linspace(0.4, 1.8, 40)

fig, ax = plt.subplots(figsize=(7, 5))
chi_peaks = []

for N in sizes:
    chis = [transverse_susceptibility(N, J=1.0, Gamma=g, n_states=min(100, 2**N-1))
            for g in gammas_chi]
    ax.plot(gammas_chi, chis, label=f'N={N}')
    chi_peaks.append(max(chis))

ax.axvline(1.0, color='red', linestyle='--', alpha=0.6, label=r'$\Gamma_c$')
ax.set_xlabel(r'$\Gamma/J$')
ax.set_ylabel(r'$\chi$')
ax.set_title('Transverse susceptibility')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Extract gamma/nu from peak scaling
log_N = np.log(sizes)
log_chi = np.log(chi_peaks)
slope, _ = np.polyfit(log_N, log_chi, 1)
print(f"\nχ_max ~ N^(γ/ν): fitted exponent = {slope:.3f} (expected γ/ν = 1.75)")

## 5. Finite-Size Scaling with ED Data

With ED results for multiple system sizes, the FSS toolkit from lecture 02 applies directly:

- **Binder cumulant crossing** locates $\Gamma_c$ without fitting.
- **$L \cdot \Delta$ crossing** provides an independent estimate.
- **Scaling collapse** extracts $\beta/\nu$ and $\nu$.

---

In [ ]:
sizes_fss = [6, 8, 10, 12, 14]
gammas_fss = np.linspace(0.5, 1.5, 60)
fss_data = {N: {'U4': [], 'gap': []} for N in sizes_fss}

for N in sizes_fss:
    Mz = sum(sz_site(i, N) for i in range(N)) / N
    for g in gammas_fss:
        H = build_tfim(N, J=1.0, Gamma=g)
        evals, evecs = eigsh(H, k=2, which='SA')
        idx = np.argsort(evals)
        psi0 = evecs[:, idx[0]]
        fss_data[N]['gap'].append(evals[idx[1]] - evals[idx[0]])
        m2 = float((psi0 @ Mz @ Mz @ psi0).real)
        m4 = float((psi0 @ Mz @ Mz @ Mz @ Mz @ psi0).real)
        fss_data[N]['U4'].append(1 - m4 / (3 * m2**2))
    print(f"N={N} done")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for N in sizes_fss:
    axes[0].plot(gammas_fss, fss_data[N]['U4'], label=f'N={N}')
    axes[1].plot(gammas_fss, N * np.array(fss_data[N]['gap']), label=f'N={N}')

for ax in axes:
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label=r'$\Gamma_c=1$')
    ax.set_xlabel(r'$\Gamma/J$')
    ax.legend(fontsize=9)

axes[0].set_ylabel(r'$U_4$')
axes[0].set_title('Binder cumulant crossing')
axes[1].set_ylabel(r'$L \cdot \Delta$')
axes[1].set_title(r'Gap crossing: $L\Delta$ vs $\Gamma/J$')

plt.tight_layout()
plt.show()

## 6. What ED Reveals about the Phase Diagram

Taken together, the diagnostics above give a complete picture of the TFIM phase diagram:

| Observable | Ordered ($\Gamma < \Gamma_c$) | Critical | Disordered ($\Gamma > \Gamma_c$) |
|---|---|---|---|
| Spectral gap $\Delta$ | $\sim e^{-N/\xi}$ | $\sim L^{-1}$ | Finite, $\sim|\Gamma-\Gamma_c|$ |
| $\langle m^2 \rangle$ | Nonzero | 0 | 0 |
| Correlations $C(r)$ | Long-range (const) | Power-law $r^{-1/4}$ | Exponential |
| Susceptibility $\chi$ | Finite | $\sim L^{7/4}$ | Finite |

The entanglement entropy (lecture 07) adds one more row: area law in both gapped phases and $\frac{1}{6}\log L$ at the critical point. It is the bridge from ED to tensor network methods.

---

In [ ]:
# Summary: phase diagram from ED observables
N = 14
gammas_pd = np.linspace(0.2, 2.5, 80)
gap_pd, m2_pd = [], []

Mz_pd = sum(sz_site(i, N) for i in range(N)) / N

for g in gammas_pd:
    H = build_tfim(N, J=1.0, Gamma=g)
    evals, evecs = eigsh(H, k=2, which='SA')
    idx = np.argsort(evals)
    psi0 = evecs[:, idx[0]]
    gap_pd.append(evals[idx[1]] - evals[idx[0]])
    m2_pd.append(float((psi0 @ Mz_pd @ Mz_pd @ psi0).real))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gammas_pd, gap_pd, 'b-')
axes[0].axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c$')
axes[0].fill_betweenx([0, max(gap_pd)], 0, 1.0, alpha=0.08, color='blue', label='Ordered')
axes[0].fill_betweenx([0, max(gap_pd)], 1.0, 2.5, alpha=0.08, color='green', label='Disordered')
axes[0].set_xlabel(r'$\Gamma/J$'); axes[0].set_ylabel(r'$\Delta$')
axes[0].set_title(f'Spectral gap, N={N}'); axes[0].legend(fontsize=9)

axes[1].plot(gammas_pd, m2_pd, 'g-')
axes[1].axvline(1.0, color='red', linestyle='--', label=r'$\Gamma_c$')
axes[1].fill_betweenx([0, max(m2_pd)], 0, 1.0, alpha=0.08, color='blue', label='Ordered')
axes[1].fill_betweenx([0, max(m2_pd)], 1.0, 2.5, alpha=0.08, color='green', label='Disordered')
axes[1].set_xlabel(r'$\Gamma/J$'); axes[1].set_ylabel(r'$\langle m^2 \rangle$')
axes[1].set_title(f'Order parameter, N={N}'); axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()